In [33]:
import os
import pdfplumber
import pypdfium2 as pyfium
import chromadb
from IPython.display import display,Markdown
from PIL import Image
from dotenv import load_dotenv
from google import genai


In [2]:
load_dotenv()
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:
db_client = chromadb.PersistentClient(
    path= "./chroma_db"
)

In [4]:
try:
    collection = db_client.get_collection(
        name = "multimodal_rag"
    )
    print("collection loaded")
except:
    collection = db_client.create_collection(
        name="multimodal_rag"
    )
    print("collection created")

collection loaded


In [37]:
def pdf_to_images(pdf_path):
    pdf =pyfium.PdfDocument(pdf_path)
    image_folder = "temp_pages"
    os.makedirs(
        image_folder,
        exist_ok= True
    )
    image_paths= []
    for i in range(len(pdf)):
        page=pdf[i]
        bitmap = page.render(scale=2)
        image = bitmap.to_pil()
        image_path = os.path.join(
            image_folder,
            f"page_{i+1}.png"
        )
        image.save(image_path)
        image_paths.append(image_path)
    return image_paths

In [38]:
images = pdf_to_images(
    "../Data/M5.pdf"
)

print(images[:3])

['temp_pages/page_1.png', 'temp_pages/page_2.png', 'temp_pages/page_3.png']


In [39]:
def analyze_image(image_path):
    image = Image.open(image_path)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            image,
            """
            Analyze this document page.

            Describe:

            - Flowcharts
            - Diagrams
            - Graphs
            - Tables
            - Visual information

            Ignore ordinary paragraph text.
            """
        ]
    )

    return response.text
    

In [41]:
description = analyze_image(
    images[0]
)

display(Markdown(description))

The document page contains a series of graphs and diagrams illustrating the K-means clustering algorithm.

**Graphs:**
There are four distinct 2D scatter plots, organized in a 2x2 grid layout under the "Working of K-Means Clustering" section, each with a descriptive title and a legend:

1.  **"Choose Initial Centroids" Graph:**
    *   **Type:** 2D Scatter Plot.
    *   **Content:** Displays numerous grey 'x' markers representing data points distributed across a coordinate plane (0.0 to 1.0 on both X and Y axes). Three larger red '*' markers are shown as randomly chosen initial centroids among these data points.
    *   **Legend:** "X Centroids", "X Data Points".

2.  **"Assign Points to Nearest Centroid" Graph:**
    *   **Type:** 2D Scatter Plot with connecting lines.
    *   **Content:** Shows the same data points, now color-coded (light blue, orange, green 'x' markers) according to their assignment to one of three clusters. The initial red '*' centroids are still present. Thin lines connect each data point to its nearest centroid, visually representing the assignment step.
    *   **Legend:** "X Centroids", "X Cluster 1", "X Cluster 2", "X Cluster 3".

3.  **"Update Centroids" Graph:**
    *   **Type:** 2D Scatter Plot.
    *   **Content:** The data points remain color-coded by cluster. The centroids have moved to new positions, now represented by larger purple '*' markers, indicating they have been recalculated as the mean of the points in their respective clusters.
    *   **Legend:** "X New Centroids", "X Cluster 1", "X Cluster 2", "X Cluster 3".

4.  **"Repeat Until Convergence" Graph:**
    *   **Type:** 2D Scatter Plot.
    *   **Content:** This graph shows the final state. Data points are still color-coded by cluster. The centroids, now represented by larger blue '*' markers, are in their final, stable positions after the iterative process has converged.
    *   **Legend:** "X Final Centroids", "X Cluster 1", "X Cluster 2", "X Cluster 3".

**Diagrams:**
Below the graphs, there is a 2x2 grid of conceptual diagrams (labeled A, B, C, D) further illustrating the K-means process:

*   **Panel A:**
    *   **Content:** A scatter of many small green circles representing data points. Three larger orange circles, labeled C1, C2, and C3, are randomly placed among them, serving as initial centroids.

*   **Panel B:**
    *   **Content:** Shows the same green data points and orange centroids (C1, C2, C3). Straight black lines connect each green data point to its closest orange centroid, depicting the assignment of points to clusters.

*   **Panel C:**
    *   **Content:** The green data points remain. The orange centroids (C1, C2, C3) have moved to new positions, representing their recalculation as the mean of their assigned data points. The connecting lines are absent, emphasizing the new centroid locations.

*   **Panel D:**
    *   **Content:** This panel illustrates the final clustering. The space is divided into three distinct regions by black lines (resembling a Voronoi diagram or decision boundaries), each containing one centroid (C1, C2, C3) and its assigned green data points, with lines connecting the points to their respective centroids. This represents the final, converged clusters.

**Flowcharts:**
There are no explicit flowcharts with standard flowchart symbols. However, the sequence of the four descriptive steps ("Choose Initial Centroids", "Assign Points to Nearest Centroid", "Update Centroids", "Repeat Until Convergence") with their corresponding graphs and the four-panel diagram implicitly depicts the iterative flow of the K-means algorithm.

**Tables:**
There are no tables present in the document.

**Visual Information (General):**
The document utilizes color-coding (red, blue, purple, green text, and colored markers/points in graphs) and bolding to highlight key terms and concepts, improving readability and emphasizing critical information about K-means clustering. The overall layout is structured with clear headings and bullet points, facilitating the understanding of the algorithm through a combination of textual explanation and visual examples.

In [44]:
def extract_visual_content(pdf_path):
    image_paths = pdf_to_images(pdf_path)
    visual_docs =[]
    for page_no, image_path in enumerate(image_paths):
        description = analyze_image(image_path)
        visual_docs.append(
            {
                "content": description,
                "type":"image",
                "page":page_no+1
            }
        )
    return visual_docs

In [46]:
visual_docs = extract_visual_content(
    "../Data/M5.pdf"
)

print(
    Markdown(visual_docs[0])
)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [6]:
def get_processed_files(collection):
    results =collection.get()
    processed_files = set()
    for metadata in results["metadatas"]:
        processed_files.add(
            metadata["source"]
        )
    return processed_files

In [7]:
def load_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    documents = []
    if extension ==".txt":
        with open(file_path,"r",encoding="utf-8") as file:
            text =file.read()
            documents.append(
                {
                    "content":text,
                    "type":"text"
                }
            )
    
    elif extension ==".pdf":
        text =""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
            
                if page_text:
                    text += page_text+ "\n"
        text = text.replace("\u2028", "\n")
        text = text.replace("\u2029", "\n")
        documents.append(
            {
                "content":text,
                "type":"text"
            }
        )
        
    else:
        raise ValueError(f"Unsupported file extension {extension}")
    return documents


In [8]:
def chunk_text(text,chunk_size=1000,overlap=200):
    chunks = []
    start =0
    while start<len(text):
        end = start+chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks    

In [9]:
def load_new_folder(folder_path,collection):
    processed_files= get_processed_files(collection)
    all_chunks = []
    for file in os.listdir(folder_path):
        if file in processed_files:
            print(f"Skipping {file} (already embedded andstored)")
            continue
        file_path = os.path.join(
            folder_path,
            file
        )
        print(f"Reading file {file}")
        
        documents = load_document(file_path)
        for doc in documents:
            chunks = chunk_text(doc["content"])
            for i,chunk in enumerate(chunks):
                all_chunks.append(
                    {
                        "source":file,
                        "chunk_id": i,
                        "content": chunk,
                        "type":doc["type"]
                    }
                )
        print(f"Loaded{file}")
    return all_chunks

In [10]:
new_chunks = load_new_folder("../Data",collection)
print(
    f"Total Chunks: {len(new_chunks)}"
)

Skipping MLDOC.txt (already embedded andstored)
Skipping M5.pdf (already embedded andstored)
Total Chunks: 0


In [15]:
def create_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents = text
    )
    return response.embeddings[0].values

In [16]:

for chunk in new_chunks:
    embedding = create_embedding(chunk["content"])
    collection.add(
        ids = [f"{chunk['source']}_{chunk['chunk_id']}"],
        documents = [chunk["content"]],
        embeddings = [embedding],
        metadatas=[
            {
                "source" : chunk["source"],
                "chunk_id" :chunk["chunk_id"],
                "type": chunk["type"]
            }
        ]
       )
print("stored successfully")


stored successfully


In [19]:
def retrieve_chunks(query):
    query_embedding = create_embedding(query)
    results = collection.query(
        query_embeddings =[query_embedding],
        n_results = 3
    )
    return{
        "documents":results["documents"][0],
        "metadata": results["documents"][0]
    }

In [21]:
results = retrieve_chunks(
    "What is machine learning?"
)

print(results["documents"][1])

What is machine learning?
Machine learning is a branch of artificial intelligence (AI) and computer science which
focuses on the use of data and algorithms to imitate the way that humans learn,
gradually improving its accuracy.
IBM has a rich history with machine learning. One of its own, Arthur Samuel, is credited
for coining the term, “machine learning” with his research (link resides outside ibm.com)
around the game of checkers. Robert Nealey, the self-proclaimed checkers master,
played the game on an IBM 7094 computer in 1962, and he lost to the computer.
Compared to what can be done today, this feat seems trivial, but it’s considered a major
milestone in the field of artificial intelligence.
Over the last couple of decades, the technological advances in storage and processing
power have enabled some innovative products based on machine learning, such as
Netflix’s recommendation engine and self-driving cars.
Machine learning is an important component of the growing field of data sc

In [26]:
def build_context(results):
    context = ""
    for doc in results["documents"][0]:
        context += doc
        context += "\n\n"
    return context

In [32]:
def ask_question(query):
    
    results = retrieve_chunks(query)
    context = build_context(results)

    # Step 5: Create prompt
    prompt = f"""
    You are a helpful AI assistant.

    Answer ONLY from the provided context.

    If the answer is not present in the context,
    say "Answer not found in document."

    Context:
    {context}

    Question:
    {query}

    Give the answer in a well-formatted manner using paragraphs and bullet points where appropriate.
    """

    # Step 6: Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text
    

In [34]:
while True:
    query = input("\nEnter a Question:")
    if query.lower()== "exit":
        break
    Answer = ask_question(query)
    display(Markdown(Answer))

K-Means Clustering is an unsupervised learning technique used to group similar data points into clusters without the need for pre-labelled data.

*   **Functionality:** It operates by grouping data points based on their distance to cluster centres, aiming to categorize items into 'k' groups or clusters of similarity. The algorithm uses Euclidean distance as a measurement for similarity.
*   **Purpose:** It is useful when there's a need to extract structure from raw, unorganized data.
*   **Applications:** Commonly employed in areas such as customer segmentation, image compression, and pattern discovery.
*   **"k" Value:** In K-Means Clustering, "k" represents the desired number of groups or clusters into which the items will be classified.